# Session 3: Building Agentic LLM Workflows for Biomedical Knowledge Graphs
Session 2 was about *how you build* a multi-omic profile. Session 3 is about *how you let something else use it*: an LLM agent — here, Claude, from Anthropic, reached through an API key — given tools so it can do more than it could using its knowledge alone.

This session walks through three parts. Part 1 lets Claude call plain Python functions directly, as tools. Part 2 moves those same tools behind a server using the MCP protocol. Part 3 gives Claude general-purpose tools instead — running code, reading and writing files — guided by written instructions rather than a fixed list of functions.

All three parts ask the model the same question, over the same tools — the same `mofa_tools.py` functions, operating on the one MOFA model fit once, back in Part 0. What changes between them is only the interface between the model and those tools — so you can see exactly what each layer (a typed tool, an MCP server, an agent skill) actually adds. The question every part answers is the same:

> Which MOFA factor is most associated with breast-cancer subtype, and what drives it?

One design choice holds across all three parts: **the model never gets raw omics data.** Every function the agent can call — check `src/mofa_tools.py` yourself — takes small typed arguments, like a factor name or a threshold, never a DataFrame. The heavy, error-prone work (aligning three separately-exported omics files, fitting MOFA, computing variance explained) happens once, in Part 0, before any agent runs. The agent's job is to reason over the results of that work, not to reproduce or verify it.

## Part 0 — Data reconciliation

Three separately-exported omics files — transcriptomics, proteomics, methylation — cover different, only partially-overlapping sets of patients, in different orders. This notebook finds the patients present in all three, checks that they agree on each patient's subtype, and merges them into one aligned `omics.pkl`. It matters specifically because of the rule above: an agent calling `load_omics_data` has no way to notice if that alignment does not hold, so someone has to guarantee it first.

## Part 1 — Tools

The basic interface: plain Python functions, each with a short description of what it does, and Claude decides on its own which one to call, and in what order, to answer a question. We do that through a library called LangChain. This is the simplest way to give a model access to real evidence instead of asking it to guess, and it is enough to see the core mechanic every agent in this session shares: send a question, let Claude request tool calls, run them, feed the results back, repeat until it answers with no more calls.

**Key takeaways**
- Handing Claude a function as a tool is not the same as trusting it with what is inside that function — the short description is the only thing Claude ever "reads" about how it works.
- `fit_mofa` is deliberately never offered as a tool. Expensive, non-deterministic operations do not belong within an agent's reach.
- Claude chooses which tools to call and in what order; nothing here hardcodes the reasoning path.

## Part 2 — The Model Context Protocol (MCP)

The tools from Part 1 only exist inside that one notebook — nobody else can use them without copying your code. MCP is a standard way to instead put tools behind a server: a small program that runs on its own, which any agent can connect to and call the same tools from, without needing your code at all.

This matters for two reasons. First, you may not want your data or your tool logic sitting inside every notebook that touches it — a server lets you keep it in one controlled place instead. Second, and more useful day-to-day: other people have already built and published tools this way. If someone else's server exposes tools over MCP, you can connect to it and use them immediately, without writing anything yourself.

We first show that the results match Part 1 exactly — same tools, same cached model, just reached through a server instead of called directly — to prove MCP changes nothing about the tools themselves, only how they are reached. Then we connect to a second, public MCP server, BioMCP, run by someone else entirely, to show the real payoff: instant access to tools over public biomedical databases (PubMed, ClinicalTrials.gov, gene and drug lookups) that we never had to build.

**Key takeaways**
- MCP does not change what a tool can do. It changes where it lives and who can reach it.
- That matters for two reasons: keeping data and logic in one controlled place, and reusing tools other people have already built and published.
- BioMCP makes the second reason concrete: connect to someone else's server and gain tools instantly, the same way you connect to your own.

## Part 3 — Agent Skills

Parts 1 and 2 both hand Claude a fixed menu of functions to choose from — whether they live in your notebook or behind an MCP server, Claude can only ever do exactly what is on that menu. Part 3 removes the menu: instead, Claude gets general-purpose tools — run code, read a file, write a file — the same tools a person would have at a terminal. With those alone, Claude could in principle do almost anything, but it would not know your specifics: which model to load, what counts as good evidence, how you want the answer formatted.

An **Agent Skill** is simply a markdown file of instructions, a `SKILL.md`, that Claude reads and follows for a task like this one, instead of you writing a separate function for every rule. Where Parts 1 and 2 defined *what Claude is allowed to do*, a skill teaches it *how to do this particular task well*.

**Key takeaways**
- Tools and MCP both hand Claude a fixed list of things it can do. A skill instead hands it instructions for using tools it already has.
- The comparison worth running yourself: same task, same model, with and without the skill loaded. What Claude *can* do does not change; how well it does it does.
- "Load the cached model, never refit" is exactly the kind of knowledge that belongs in a skill, not a function signature — it is a rule about how to use a tool, not a new tool.